# IRIS Phase 2: DistilRoBERTa fine-tuning on a Colab GPU

Runs the **same experiments** as the project (`e3`, `e3s`, `e3o`, plus `e3` with the lenient mapping) using the same dataset, label
mapping, frozen split, 5-fold CV folds, calibration and metrics. Only the compute moves to Colab. The script **refuses to start**
if the data, labels, split or folds differ from your local project, and it has **no CPU fallback**.

Expected time on a T4: roughly 1 to 1.5 hours. It is resumable: finished experiments are skipped if you run it again, so keep the
output folder on Google Drive.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. You need `iris_colab_bundle.zip` (built on your laptop with
`python colab/build_bundle.py`). Nothing from the bundle is printed or shared. The Reddit data is used under CC-BY-4.0.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Drive folder for the results (survives disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/iris_phase2_colab'
import os; os.makedirs(OUT, exist_ok=True)
print('results will be written to', OUT)

## 3. Get the bundle
If `iris_colab_bundle.zip` is in the root of your Drive it is used; otherwise a file picker opens.

In [ ]:
import os, zipfile, shutil
DRIVE_ZIP = '/content/drive/MyDrive/iris_colab_bundle.zip'
if os.path.exists(DRIVE_ZIP):
    zip_path = DRIVE_ZIP
else:
    from google.colab import files
    up = files.upload()
    zip_path = next(iter(up))
shutil.rmtree('/content/iris', ignore_errors=True)
zipfile.ZipFile(zip_path).extractall('/content/iris')
%cd /content/iris/ml
!ls

## 4. (Optional) ESConv for the false-alarm audit
If your bundle was built **without** `--include-esconv`, you can upload your own copy of `ESConv.json` here (academic-research-only
licence: used **only** to count false alarms on ordinary chat, never for training). Skip this cell to skip that audit.

In [ ]:
import os
if not os.path.exists('data/raw/esconv/ESConv.json'):
    from google.colab import files
    up = files.upload()          # choose ESConv.json, or cancel to skip
    if up:
        os.makedirs('data/raw/esconv', exist_ok=True)
        shutil.move(next(iter(up)), 'data/raw/esconv/ESConv.json')
print('ESConv present:', os.path.exists('data/raw/esconv/ESConv.json'))

## 5. Install the few extras and verify the bundle (trains nothing)

In [ ]:
!pip install -q -r requirements-colab.txt
!python colab/colab_finetune.py --out {OUT} --dry-run

## 6. Train and evaluate
Prints a line for every fold. All numbers come from the model's own out-of-fold predictions.

In [ ]:
!python -u colab/colab_finetune.py --out {OUT}

## 7. Results

In [ ]:
print(open(f'{OUT}/finetune_summary.md').read())

## 8. Download
`iris_colab_results.zip` holds the results (small). The model weights stay in `models/` inside the Drive folder (about 330 MB each).
Put the zip into the project and run `python colab/ingest_results.py <zip>`, then `python -m iris_ml.run_phase2 report`.

In [ ]:
from google.colab import files
files.download(f'{OUT}/iris_colab_results.zip')